# iNaturalist Species Download

Downloads research-grade wildlife images from iNaturalist for all 7 target species.
Images are saved to Google Drive at `MyDrive/bc-dashcam/data/<species>/`.

**Species:** moose, elk, bear, cougar, deer, coyote, raccoon
**Source:** iNaturalist BC (place_id=7085), except cougar which uses global observations.
**Cap:** 500 images per species.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pyinaturalist -q

In [ ]:
import requests
import os
from pyinaturalist import get_observations
import time
import json


def download_species(taxon_name, place_id, species_label, max_images=500, global_search=False):
    save_dir = f"/content/drive/MyDrive/bc-dashcam/data/{species_label}"
    os.makedirs(save_dir, exist_ok=True)

    existing = set(os.listdir(save_dir))
    downloaded = 0
    page = 1
    metadata = []

    while downloaded < max_images:
        print(f"{species_label} - Fetching page {page}...")
        try:
            kwargs = dict(
                taxon_name=taxon_name,
                quality_grade="research",
                photos=True,
                per_page=50,
                page=page
            )
            if not global_search:
                kwargs['place_id'] = place_id

            results = get_observations(**kwargs)
        except Exception as e:
            print(f"API call failed: {e}")
            break

        if not results['results']:
            print("No more results.")
            break

        for obs in results['results']:
            if downloaded >= max_images:
                break
            try:
                photo_url = obs['photos'][0]['url'].replace("square", "large")
                obs_id = obs['id']
                filename = f"{obs_id}.jpg"

                if filename in existing:
                    downloaded += 1
                    continue

                response = requests.get(photo_url, timeout=10)
                if response.status_code == 200:
                    with open(f"{save_dir}/{filename}", 'wb') as f:
                        f.write(response.content)
                    metadata.append({
                        'obs_id': obs_id,
                        'taxon': taxon_name,
                        'label': species_label,
                        'url': photo_url
                    })
                    downloaded += 1
                    if downloaded % 100 == 0:
                        print(f"  {species_label}: {downloaded}/{max_images}")
                time.sleep(0.3)
            except Exception as e:
                print(f"Error on {obs['id']}: {e}")
                continue

        page += 1
        time.sleep(1)

    meta_path = f"/content/drive/MyDrive/bc-dashcam/metadata/{species_label}_metadata.json"
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Done: {species_label} - {downloaded} images saved to Drive.")
    return metadata

## Download all 7 species

Run each cell below. Cougar uses `global_search=True` because BC-only observations (332) are insufficient.

In [ ]:
download_species(
    taxon_name="Alces alces",
    place_id=7085,
    species_label="moose",
    max_images=500
)

In [ ]:
download_species(
    taxon_name="Cervus canadensis",
    place_id=7085,
    species_label="elk",
    max_images=500
)

In [ ]:
download_species(
    taxon_name="Ursus americanus",
    place_id=7085,
    species_label="bear",
    max_images=500
)

In [ ]:
# Global search: BC has only 332 cougar observations
download_species(
    taxon_name="Puma concolor",
    place_id=None,
    species_label="cougar",
    max_images=500,
    global_search=True
)

In [ ]:
download_species(
    taxon_name="Odocoileus hemionus",
    place_id=7085,
    species_label="deer",
    max_images=500
)

In [ ]:
download_species(
    taxon_name="Canis latrans",
    place_id=7085,
    species_label="coyote",
    max_images=500
)

In [ ]:
download_species(
    taxon_name="Procyon lotor",
    place_id=7085,
    species_label="raccoon",
    max_images=500
)

## Verify download counts

In [ ]:
import os

species = ['moose', 'elk', 'bear', 'cougar', 'deer', 'coyote', 'raccoon']
for s in species:
    path = f"/content/drive/MyDrive/bc-dashcam/data/{s}"
    if os.path.exists(path):
        print(f"{s}: {len(os.listdir(path))} images")
    else:
        print(f"{s}: folder missing")